# Reinforcement Learning — Q-Learning on FrozenLake

A complete introduction to **tabular Reinforcement Learning**: an agent learns to cross a
frozen lake (FrozenLake-v1) from scratch using **Q-learning** — no neural networks,
just the Bellman equation.

**Approach:**
1. Meet the environment (states, actions, rewards)
2. Random policy baseline
3. Q-learning training with ε-greedy exploration
4. Evaluation & learned-policy visualization
5. Slippery vs. non-slippery comparison

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym

sns.set_style("whitegrid")
rng = np.random.default_rng(42)
print("gymnasium version:", gym.__version__)

gymnasium version: 1.4.0


## 1. The Environment — FrozenLake-v1

A 4×4 grid world. The agent starts at **S**, must reach the goal **G**, and avoid holes **H**.

```
S F F F
F H F H
F F F H
H F F G
```

- **States:** 16 (one per tile) · **Actions:** 4 (Left, Down, Right, Up)
- **Reward:** +1 for reaching G, 0 otherwise — sparse and delayed
- **Slippery ice:** with `is_slippery=True`, the agent moves in the intended
  direction only 1/3 of the time — the environment is *stochastic*

In [2]:
env = gym.make('FrozenLake-v1', desc=None, map_name='4x4', is_slippery=True,
               render_mode='ansi')

n_states = env.observation_space.n
n_actions = env.action_space.n
print(f"States: {n_states}, Actions: {n_actions}")
print("Action meanings: 0=Left  1=Down  2=Right  3=Up")
env.reset(seed=42)
print(env.render())  # ANSI grid

States: 16, Actions: 4
Action meanings: 0=Left  1=Down  2=Right  3=Up

SFFF
FHFH
FFFH
HFFG



## 2. Baseline — Random Policy

How well does pure chance do? This is the number Q-learning has to beat.

In [3]:
def run_episode(env, policy_fn, max_steps=100):
    """Run one episode; policy_fn(state) -> action. Returns total reward."""
    state, _ = env.reset()
    total_reward = 0
    for _ in range(max_steps):
        state, reward, terminated, truncated, _ = env.step(policy_fn(state))
        total_reward += reward
        if terminated or truncated:
            break
    return total_reward

random_rewards = [run_episode(env, lambda s: env.action_space.sample())
                  for _ in range(1000)]
print(f"Random policy success rate: {np.mean(random_rewards):.3f}  (over 1000 episodes)")

Random policy success rate: 0.011  (over 1000 episodes)


## 3. Q-Learning

Maintain a table **Q(s, a)** = expected return of taking action *a* in state *s*, and update it
after every step with the **Bellman equation**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big]$$

- **α (learning rate)** — how fast new information overrides old
- **γ (discount factor)** — how much future rewards matter
- **ε-greedy exploration** — with probability ε take a random action, else exploit
  the best known action; ε decays over time as the agent becomes confident

In [4]:
def train_q_learning(env, episodes=10_000, alpha=0.1, gamma=0.99,
                     epsilon=1.0, min_epsilon=0.01, decay_rate=0.9995, seed=42):
    rng = np.random.default_rng(seed)   # local RNG: reproducible regardless of cell order
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    rewards_history, eps_history = [], []
    eps = epsilon

    for ep in range(episodes):
        state, _ = env.reset(seed=seed + ep)
        done = False
        total = 0
        while not done:
            if rng.random() < eps:                    # explore
                action = env.action_space.sample()
            else:                                     # exploit
                action = np.argmax(Q[state])

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Bellman update
            target = reward + gamma * np.max(Q[next_state]) * (not terminated)
            Q[state, action] += alpha * (target - Q[state, action])

            state = next_state
            total += reward

        eps = max(min_epsilon, eps * decay_rate)
        rewards_history.append(total)
        eps_history.append(eps)

    return Q, rewards_history, eps_history

Q, rewards_history, eps_history = train_q_learning(env)
print(f"Training done — success rate over last 500 episodes: "
      f"{np.mean(rewards_history[-500:]):.3f}")

Training done — success rate over last 500 episodes: 0.712


## 4. Training Curves

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

window = 500
moving_avg = np.convolve(rewards_history, np.ones(window) / window, mode='valid')
axes[0].plot(moving_avg)
axes[0].set_title(f'Success Rate (moving average, window={window})')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Success rate')

axes[1].plot(eps_history)
axes[1].set_title('Exploration Rate (ε) Decay')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('ε')

plt.tight_layout()
plt.show()

<Figure size 1300x400 with 2 Axes>

## 5. Evaluation — Trained Agent vs. Random

In [6]:
greedy = lambda s: np.argmax(Q[s])
trained_rewards = [run_episode(env, greedy) for _ in range(1000)]

print(f"Random policy  : {np.mean(random_rewards):.3f}")
print(f"Trained policy : {np.mean(trained_rewards):.3f}")

plt.figure(figsize=(6, 4))
plt.bar(['Random', 'Q-learning (greedy)'],
        [np.mean(random_rewards), np.mean(trained_rewards)],
        color=['#d62728', '#2ca02c'])
plt.ylabel('Success rate over 1000 episodes')
plt.title('FrozenLake 4x4 — Random vs. Learned Policy')
plt.show()

Random policy  : 0.011
Trained policy : 0.728


<Figure size 600x400 with 1 Axes>

## 6. Visualize the Learned Policy

Arrows show the greedy action at each tile. On **slippery** ice the optimal policy often
takes longer paths that keep distance from holes — the agent has learned *risk avoidance*.

In [7]:
arrow = {0: '←', 1: '↓', 2: '→', 3: '↑'}
desc = env.unwrapped.desc.astype(str)   # tile types: S F H G
policy_grid = np.full(desc.shape, ' ', dtype=object)

for s in range(n_states):
    r, c = divmod(s, desc.shape[1])
    tile = desc[r, c]
    policy_grid[r, c] = tile if tile in ('S', 'H', 'G') else arrow[np.argmax(Q[s])]

fig, ax = plt.subplots(figsize=(5, 5))
colors = {'S': '#9ecae1', 'F': '#f7fbff', 'H': '#252525', 'G': '#a1d99b'}
for r in range(desc.shape[0]):
    for c in range(desc.shape[1]):
        ax.add_patch(plt.Rectangle((c, -r), 1, 1,
                     facecolor=colors[desc[r, c]], edgecolor='gray'))
        ax.text(c + 0.5, -r - 0.5, policy_grid[r, c],
                ha='center', va='center', fontsize=18,
                color='white' if desc[r, c] == 'H' else 'black')
ax.set_xlim(0, 4); ax.set_ylim(-4, 0); ax.axis('off')
ax.set_title('Learned Policy (S=start, G=goal, H=hole)')
plt.show()

<Figure size 500x500 with 1 Axes>

## 7. Watch One Episode Step by Step

In [8]:
env_show = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True,
                    render_mode='ansi')
state, _ = env_show.reset(seed=7)
print(env_show.render())
done, step = False, 0
while not done and step < 20:
    action = int(np.argmax(Q[state]))          # greedy action from the Q-table
    state, reward, terminated, truncated, _ = env_show.step(action)
    done = terminated or truncated
    step += 1
    print(f"Step {step}: took action {arrow[action]}")
    print(env_show.render())
print("Reached the goal! 🏆" if reward == 1 else "Fell in a hole ❄️")


SFFF
FHFH
FFFH
HFFG

Step 1: took action ←
  (Left)
SFFF
FHFH
FFFH
HFFG

Step 2: took action ←
  (Left)
SFFF
FHFH
FFFH
HFFG

Step 3: took action ↑
  (Up)
SFFF
FHFH
FFFH
HFFG

Step 4: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Step 5: took action ↑
  (Up)
SFFF
FHFH
FFFH
HFFG

Step 6: took action ↑
  (Up)
SFFF
FHFH
FFFH
HFFG

Step 7: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Step 8: took action ←
  (Left)
SFFF
FHFH
FFFH
HFFG

Step 9: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Step 10: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Step 11: took action →
  (Right)
SFFF
FHFH
FFFH
HFFG

Step 12: took action →
  (Right)
SFFF
FHFH
FFFH
HFFG

Step 13: took action →
  (Right)
SFFF
FHFH
FFFH
HFFG

Step 14: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Step 15: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Step 16: took action ↓
  (Down)
SFFF
FHFH
FFFH
HFFG

Reached the goal! 🏆


## 8. Slippery vs. Non-Slippery

The same algorithm on a deterministic lake (`is_slippery=False`) converges to a ~100%
success rate — stochasticity is what makes the problem hard.

In [9]:
env_det = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)
Q_det, rew_det, _ = train_q_learning(env_det, episodes=5000, decay_rate=0.999)

det_rewards = [run_episode(env_det, lambda s: np.argmax(Q_det[s])) for _ in range(1000)]
print(f"Deterministic lake — trained success rate: {np.mean(det_rewards):.3f}")

window = 200
plt.figure(figsize=(7, 4))
plt.plot(np.convolve(rewards_history, np.ones(window)/window, mode='valid'),
         label='Slippery (stochastic)')
plt.plot(np.convolve(rew_det, np.ones(window)/window, mode='valid'),
         label='Non-slippery (deterministic)')
plt.xlabel('Episode'); plt.ylabel(f'Success rate (MA, window={window})')
plt.title('Q-Learning Convergence: Stochastic vs. Deterministic')
plt.legend(); plt.show()

Deterministic lake — trained success rate: 1.000


<Figure size 700x400 with 1 Axes>

## 9. Conclusion

- **Q-learning** learns an optimal policy purely from trial and error — no model of the
  environment needed (*model-free*).
- **ε-greedy decay** balances exploration vs. exploitation: random early, greedy late.
- **Stochasticity** (slippery ice) caps achievable performance and forces risk-averse
  paths — the same algorithm reaches ~100% on a deterministic map.
- Next steps: **SARSA** (on-policy), **n-step** returns, scaling to continuous states with
  **Deep Q-Networks (DQN)** on CartPole.